# Solución del caso práctico 1
# Temas 1 y 2: fundamentos + criptografía asimétrica

## 1) Cifrado simétrico

Usamos el esquema `C = (M + K) mod 26` con `K = 13`.

- H -> 7 + 13 = 20 -> U
- E -> 4 + 13 = 17 -> R
- L -> 11 + 13 = 24 -> Y
- L -> 24 -> Y
- O -> 14 + 13 = 27 -> 1 -> B

Resultado: `URYYB`.

## 2) Qué protege cada mecanismo

- El cifrado protege la confidencialidad: si no se conoce la clave, el mensaje no es legible.
- El hash protege la integridad: una mínima modificación del mensaje cambia totalmente la huella digital.
- La integridad no implica autenticación por sí sola; quien envía y quien recibe deben compartir o validar la clave de manera segura.

## 3) Qué ocurre si se modifica el ciphertext

Si un atacante altera el texto cifrado y el receptor lo descifra con la clave correcta, el resultado será un texto plano que normalmente no tendrá sentido. Además, si se compara con la firma o el hash esperado, la comparación fallará, demostrando que el contenido no es fiable.

## 4) Simétrica frente a asimétrica

La criptografía simétrica es muy eficiente para cifrar grandes volúmenes de datos, pero exige compartir una clave secreta antes del intercambio. La asimétrica resuelve el problema de distribución de claves: se usa una clave pública para cifrar o verificar y una privada para descifrar o firmar.

En la práctica, se suele combinar ambos enfoques: intercambio de claves asimétricas y cifrado simétrico del contenido real.

## 5) Ejemplo RSA sencillo

Con `p = 5`, `q = 11`, `n = 55`, `phi(n) = 40` y `e = 3`, calculamos `d` como el inverso de `e` módulo `40`, que es `27`, porque `3 * 27 = 81 = 1 mod 40`.

Entonces, si `M = 7`,

$$C = M^e mod n = 7^3 mod 55 = 343 mod 55 = 38$$

y al descifrar,

$$M = C^d mod n = 38^{27} mod 55 = 7$$

## 6) ¿Se puede deducir la clave a partir del texto cifrado?

No, al menos no de forma inmediata. El cifrado con desplazamiento es una transformación determinista sobre un alfabeto conocido, pero sin la clave no es posible reconstruir el desplazamiento exacto. En la práctica, aunque el atacante pueda detectar el patrón del cifrado, necesita probar un número limitado de claves, algo que no es viable si la clave es suficientemente grande.

## 7) Protocolo de intercambio

Una forma sencilla de hacerlo es: Alice genera una clave simétrica `K`, la cifra con la clave pública de Bob y la envía. Bob la descifra con su clave privada. Después, ambos usan `K` para cifrar el contenido del mensaje con un esquema simétrico eficiente. Esto añade la ventaja de no depender de la asimetría para cifrar todo el tráfico.

## 8) Conclusión

El caso demuestra que la seguridad de un sistema no depende solo de un algoritmo concreto, sino de la combinación adecuada de mecanismos: cifrado para confidencialidad, hash para integridad y protocolos de clave para distribuir secretos sin exponerse a un ataque de intermediario.

## 9) Implementación de ejemplo

In [ ]:
import hashlib

alphabet = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def letter_to_index(ch):
    return alphabet.index(ch.upper())

def index_to_letter(i):
    return alphabet[i % 26]

def encrypt_caesar(message, key):
    cipher = []
    for ch in message.upper():
        if ch.isalpha():
            idx = letter_to_index(ch)
            cipher.append(index_to_letter(idx + key))
        else:
            cipher.append(ch)
    return ''.join(cipher)

def decrypt_caesar(cipher, key):
    return encrypt_caesar(cipher, -key)

message = 'HELLO'
key = 13
cipher = encrypt_caesar(message, key)
plain = decrypt_caesar(cipher, key)
print('Mensaje:', message)
print('Cifrado:', cipher)
print('Descifrado:', plain)

hash_original = hashlib.sha256(message.encode()).hexdigest()
hash_modificado = hashlib.sha256('HELLO!'.encode()).hexdigest()
print('Hash original:', hash_original)
print('Hash modificado:', hash_modificado)
print('Son iguales:', hash_original == hash_modificado)

## 5) Conclusión

El esquema de cifrado ofrece confidencialidad, pero no autenticación ni integridad por sí solo. Para que el sistema sea seguro en la práctica, hace falta una detección de cambios y una autenticación del remitente.

En la práctica, esto se resuelve con un cifrado autenticado (AEAD) y una firma digital o MAC, según el caso.